<h1 style="color:green;font-size:22px;">Function 8 - Black-Box Optimisation</h1>
<h1 style="color:#0000CD;font-size:19px;"">Introduction and illustrative analogy</h1>

**Function 8** is an eight-dimensional black-box objective defined over the bounded domain $[0,1]^8$. Its analytical form and underlying interpretation are unknown. Each input variable may influence the observed output, but the internal relationships, interactions and degree of non-linearity are not available to the optimiser. The objective is to identify the input configuration that maximises **Function 8** under a limited sequential-query budget. 

A useful analogy is the tuning of eight machine-learning hyperparameters (such as learning rate, batch size, number of layers, dropout rate, regularisation strength, activation function, optimiser type and initial weight range). Each input vector represents one candidate configuration, and the function returns a single performance score.

In [1]:
import numpy as np
import pandas as pd
import scipy.stats as s
import matplotlib.pyplot as plt
import warnings

from sklearn.exceptions import ConvergenceWarning

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, WhiteKernel, ConstantKernel
warnings.filterwarnings("ignore", category=ConvergenceWarning)

<h1 style="color:#0000CD;font-size:19px;"">Week 1</h1>

**1.1 - Extraction of Initial Data**

In [2]:
inputs = np.load('Initial Data/function_8/initial_inputs.npy')
outputs = np.load('Initial Data/function_8/initial_outputs.npy')
print(inputs.shape, outputs.shape)

(40, 8) (40,)


In [3]:
data = pd.DataFrame(inputs, columns=['x1','x2','x3','x4','x5','x6','x7','x8'])
data['y'] = outputs
display(data)

,x1,x2,x3,x4,x5,x6,x7,x8,y
0,0.604994,0.292215,0.908453,0.355506,0.201669,0.575338,0.310311,0.734281,7.398721
1,0.178007,0.566223,0.994862,0.210325,0.320153,0.707909,0.635384,0.107132,7.005227
2,0.009077,0.811626,0.520520,0.075687,0.265112,0.091652,0.592415,0.367320,8.459482
3,0.506028,0.653730,0.363411,0.177981,0.093728,0.197425,0.755827,0.292472,8.284008
4,0.359909,0.249076,0.495997,0.709215,0.114987,0.289207,0.557295,0.593882,8.606117
5,0.778818,0.003419,0.337983,0.519528,0.820907,0.537247,0.551347,0.660032,8.541748
6,0.908649,0.062250,0.238260,0.766604,0.132336,0.990244,0.688068,0.742496,7.327435
7,0.586371,0.880736,0.745021,0.546035,0.009649,0.748992,0.230907,0.097916,7.299872
8,0.761137,0.854672,0.382124,0.337352,0.689708,0.309853,0.631380,0.041956,7.957875
9,0.984933,0.699506,0.998885,0.180148,0.580143,0.231087,0.490827,0.313683,5.592193


In [4]:
best = outputs.max()
field = outputs.max() - outputs.min()
print(best, field)

9.598482002566342 4.006288613026146


**1.2 - Optimisation (GP + UCB)**

With only 40 initial observations in eight dimensions, a Gaussian Process surrogate with an RBF kernel is therefore combined with Upper Confidence Bound acquisition to balance predicted performance and model uncertainty.

In [5]:
# Cartesian Candidate grid for candidate selection
x1 = np.linspace(0,1,5)
x2 = np.linspace(0,1,5)
x3 = np.linspace(0,1,5)
x4 = np.linspace(0,1,5)
x5 = np.linspace(0,1,5)
x6 = np.linspace(0,1,5)
x7 = np.linspace(0,1,5)
x8 = np.linspace(0,1,5)

xx1, xx2, xx3, xx4, xx5, xx6, xx7, xx8 = np.meshgrid(x1, x2, x3, x4, x5, x6, x7, x8)
X_grid = np.column_stack([xx1.ravel(), xx2.ravel(), xx3.ravel(), xx4.ravel(), xx5.ravel(), xx6.ravel(),xx7.ravel(),xx8.ravel()])
del xx1, xx2, xx3, xx4, xx5, xx6, xx7, xx8

# Predict GP mean and uncertainty
kernel = RBF(length_scale=0.2) + WhiteKernel(noise_level=1e-6)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-10, normalize_y=True)
gp.fit(inputs, outputs)
y_pred, sigma = gp.predict(X_grid, return_std=True)

# Upper Confidence Bound
kappa = 1.5
ucb = y_pred + kappa * sigma
index_max = np.argmax(ucb)
next_point = np.array(X_grid[index_max], float)
print(f"Next point (with GP+UCB):{next_point[0]:.6f}-{next_point[1]:.6f}-{next_point[2]:.6f}-{next_point[3]:.6f}-{next_point[4]:.6f}-{next_point[5]:.6f}-{next_point[6]:.6f}-{next_point[7]:.6f}")


Next point (with GP+UCB):0.250000-0.250000-0.000000-0.000000-0.750000-0.250000-0.000000-0.500000


<h1 style="color:#0000CD;font-size:19px;"">Week 2</h1>

**2.1 - Previous Week's Query Result**

In [6]:
# New Query Point
x_new = np.array([[0.25, 0.25, 0.00, 0.00, 0.75, 0.25, 0.00, 0.50]])
y_new = 9.72705

def add_QueriedPoint(data, x_new, y_new):

    inputs = data[['x1', 'x2','x3', 'x4','x5','x6','x7','x8']].to_numpy()
    outputs = data['y'].to_numpy()
    
    # Checks if New Points is already included in data
    exists = False
    for i in range(inputs.shape[0]):
        if np.allclose(inputs[i], x_new) and np.isclose(outputs[i], y_new):
            exists = True
            break

    # Only adds if it doesn't exist already 
    if not exists:
        inputs = np.vstack([inputs, x_new])
        outputs = np.append(outputs, y_new)
        data = pd.DataFrame(inputs, columns=['x1', 'x2','x3', 'x4','x5','x6','x7','x8']).assign(y=outputs)
    
        print("Point added!")
    else:
        print("Point already exists, skipping addition.")
    return data

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4','x5','x6','x7','x8']].to_numpy()
outputs = data['y'].to_numpy()

display(data)

Point added!


,x1,x2,x3,x4,x5,x6,x7,x8,y
0,0.604994,0.292215,0.908453,0.355506,0.201669,0.575338,0.310311,0.734281,7.398721
1,0.178007,0.566223,0.994862,0.210325,0.320153,0.707909,0.635384,0.107132,7.005227
2,0.009077,0.811626,0.520520,0.075687,0.265112,0.091652,0.592415,0.367320,8.459482
3,0.506028,0.653730,0.363411,0.177981,0.093728,0.197425,0.755827,0.292472,8.284008
4,0.359909,0.249076,0.495997,0.709215,0.114987,0.289207,0.557295,0.593882,8.606117
5,0.778818,0.003419,0.337983,0.519528,0.820907,0.537247,0.551347,0.660032,8.541748
6,0.908649,0.062250,0.238260,0.766604,0.132336,0.990244,0.688068,0.742496,7.327435
7,0.586371,0.880736,0.745021,0.546035,0.009649,0.748992,0.230907,0.097916,7.299872
8,0.761137,0.854672,0.382124,0.337352,0.689708,0.309853,0.631380,0.041956,7.957875
9,0.984933,0.699506,0.998885,0.180148,0.580143,0.231087,0.490827,0.313683,5.592193


In [7]:
best = outputs.max()
field = outputs.max() - outputs.min()
print(best, field)

9.72705 4.134856610459804


**2.2 - Next Query**

In [8]:
# Reuse of 8D Cartesion grid

# GP Fit First
kernel = RBF(length_scale=0.2) + WhiteKernel(noise_level=1e-6)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-10, normalize_y=True)
gp.fit(inputs, outputs)
y_pred, sigma = gp.predict(X_grid, return_std=True)

# Upper Confidence Bound for exploration (optional, still using kappa)
kappa = 1.5
ucb = y_pred + kappa * sigma
index_ucb = np.argmax(ucb)
next_point_ucb = X_grid[index_ucb]

# Continue exploration using UCB
print("Continue UCB exploration")
next_point = next_point_ucb
print(f"Next point (UCB): {next_point}")

Continue UCB exploration
Next point (UCB): [0.   0.25 0.   0.5  0.75 0.5  0.25 0.5 ]


<h1 style="color:#0000CD;font-size:19px;"">Week 3</h1>

**3.1 - Previous Week's Query Result**

In [9]:
# New Query Point
x_new = np.array([[0.00, 0.25, 0.00, 0.50, 0.75, 0.50, 0.25, 0.50]])
y_new = 9.78955

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4','x5','x6','x7','x8']].to_numpy()
outputs = data['y'].to_numpy()

display(data)

Point added!


,x1,x2,x3,x4,x5,x6,x7,x8,y
0,0.604994,0.292215,0.908453,0.355506,0.201669,0.575338,0.310311,0.734281,7.398721
1,0.178007,0.566223,0.994862,0.210325,0.320153,0.707909,0.635384,0.107132,7.005227
2,0.009077,0.811626,0.520520,0.075687,0.265112,0.091652,0.592415,0.367320,8.459482
3,0.506028,0.653730,0.363411,0.177981,0.093728,0.197425,0.755827,0.292472,8.284008
4,0.359909,0.249076,0.495997,0.709215,0.114987,0.289207,0.557295,0.593882,8.606117
5,0.778818,0.003419,0.337983,0.519528,0.820907,0.537247,0.551347,0.660032,8.541748
6,0.908649,0.062250,0.238260,0.766604,0.132336,0.990244,0.688068,0.742496,7.327435
7,0.586371,0.880736,0.745021,0.546035,0.009649,0.748992,0.230907,0.097916,7.299872
8,0.761137,0.854672,0.382124,0.337352,0.689708,0.309853,0.631380,0.041956,7.957875
9,0.984933,0.699506,0.998885,0.180148,0.580143,0.231087,0.490827,0.313683,5.592193


In [10]:
best = outputs.max()
field = outputs.max() - outputs.min()
print(best, field)

9.78955 4.197356610459804


**3.2 - Next Query**

Broader exploration is encouraged at this stage. We increase $kappa$.

In [11]:
# GP Fit First
kernel = RBF(length_scale=0.2) + WhiteKernel(noise_level=1e-6)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-10, normalize_y=True)
gp.fit(inputs, outputs)
y_pred, sigma = gp.predict(X_grid, return_std=True)

# Upper Confidence Bound for exploration (optional, still using kappa)
kappa = 2.57
ucb = y_pred + kappa * sigma
index_ucb = np.argmax(ucb)
next_point_ucb = X_grid[index_ucb]

# Continue exploration using UCB
print("Continue UCB exploration")
next_point = next_point_ucb
print(f"Next point (UCB): {next_point}")

Continue UCB exploration
Next point (UCB): [0.   0.75 0.   0.   0.5  1.   0.   1.  ]


<h1 style="color:#0000CD;font-size:19px;"">Week 4</h1>

**4.1 - Previous Week's Query Result**

In [12]:
# New Query Point
x_new = np.array([[0.00, 0.75, 0.00, 0.00, 0.5, 1.00, 0.00, 1]])
y_new = 9.1558

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4','x5','x6','x7','x8']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

Point added!


,x1,x2,x3,x4,x5,x6,x7,x8,y
0,0.604994,0.292215,0.908453,0.355506,0.201669,0.575338,0.310311,0.734281,7.398721
1,0.178007,0.566223,0.994862,0.210325,0.320153,0.707909,0.635384,0.107132,7.005227
2,0.009077,0.811626,0.520520,0.075687,0.265112,0.091652,0.592415,0.367320,8.459482
3,0.506028,0.653730,0.363411,0.177981,0.093728,0.197425,0.755827,0.292472,8.284008
4,0.359909,0.249076,0.495997,0.709215,0.114987,0.289207,0.557295,0.593882,8.606117
5,0.778818,0.003419,0.337983,0.519528,0.820907,0.537247,0.551347,0.660032,8.541748
6,0.908649,0.062250,0.238260,0.766604,0.132336,0.990244,0.688068,0.742496,7.327435
7,0.586371,0.880736,0.745021,0.546035,0.009649,0.748992,0.230907,0.097916,7.299872
8,0.761137,0.854672,0.382124,0.337352,0.689708,0.309853,0.631380,0.041956,7.957875
9,0.984933,0.699506,0.998885,0.180148,0.580143,0.231087,0.490827,0.313683,5.592193


In [13]:
best = outputs.max()
field = outputs.max() - outputs.min()
print(best, field)

9.78955 4.197356610459804


**4.2 - Next Point Query Selection: UCB with Cartesian Grid**

In [14]:
# GP Fit First
kernel = RBF(length_scale=0.2) + WhiteKernel(noise_level=1e-6)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-10, normalize_y=True)
gp.fit(inputs, outputs)
y_pred, sigma = gp.predict(X_grid, return_std=True)

# Upper Confidence Bound for exploration (optional, still using kappa)
kappa = 2.57
ucb = y_pred + kappa * sigma
index_ucb = np.argmax(ucb)
next_point_ucb = X_grid[index_ucb]

# Continue exploration using UCB
print("For diagnostic purposes only")
next_point = next_point_ucb
print(f"Next point (UCB+Grid): {next_point}")

For diagnostic purposes only
Next point (UCB+Grid): [0.   0.   0.5  0.   1.   0.75 0.   0.75]


**Observation:** Successive grid-based UCB proposals are concentrated in a similar region. In eight dimensions, materially increasing the resolution of the Cartesian grid would cause the number of candidate points to grow rapidly. To improve candidate-space coverage without expanding the full tensor grid, the next iteration makes two changes:

- replace the Cartesian-grid candidates with a scrambled Sobol sequence;
- replace the RBF kernel with a Matérn kernel using $\nu=2.5$.

The acquisition rule remains UCB for this iteration.

**4.3 - Alternative Query Selection: UCB with Sobol Candidates**

In [15]:
from scipy.stats import qmc

d = inputs.shape[1]
m = 13
sampler = qmc.Sobol(d=d, scramble=True, seed=42)
X_cand = sampler.random_base2(m)

# GP Fit First
kernel = Matern(length_scale=[0.2]*d, nu=2.5) + WhiteKernel(noise_level=1e-5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-10, normalize_y=True)
gp.fit(inputs, outputs)
y_pred, sigma = gp.predict(X_cand, return_std=True)

# Upper Confidence Bound for exploration (optional, still using kappa)
kappa = 2.57
ucb = y_pred + kappa * sigma
index_ucb = np.argmax(ucb)
next_point_ucb = X_cand[index_ucb]

# Continue exploration using UCB
print("Submitted Query")
next_point = next_point_ucb
print(f"Next point (UCB+Sobol): {next_point}")

Submitted Query
Next point (UCB+Sobol): [0.0706096  0.12232586 0.25269797 0.02014378 0.84929346 0.95176367
 0.12203504 0.11291862]


<h1 style="color:#0000CD;font-size:19px;"">Week 5</h1>

**5.1 - Previous Week's Query Result**

In [16]:
# New Query Point
x_new = np.array([[0.070610, 0.122326, 0.252698, 0.020144, 0.849293, 0.951764, 0.122035, 0.112919]])
y_new = 9.6942921430494

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4','x5','x6','x7','x8']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

Point added!


,x1,x2,x3,x4,x5,x6,x7,x8,y
0,0.604994,0.292215,0.908453,0.355506,0.201669,0.575338,0.310311,0.734281,7.398721
1,0.178007,0.566223,0.994862,0.210325,0.320153,0.707909,0.635384,0.107132,7.005227
2,0.009077,0.811626,0.520520,0.075687,0.265112,0.091652,0.592415,0.367320,8.459482
3,0.506028,0.653730,0.363411,0.177981,0.093728,0.197425,0.755827,0.292472,8.284008
4,0.359909,0.249076,0.495997,0.709215,0.114987,0.289207,0.557295,0.593882,8.606117
5,0.778818,0.003419,0.337983,0.519528,0.820907,0.537247,0.551347,0.660032,8.541748
6,0.908649,0.062250,0.238260,0.766604,0.132336,0.990244,0.688068,0.742496,7.327435
7,0.586371,0.880736,0.745021,0.546035,0.009649,0.748992,0.230907,0.097916,7.299872
8,0.761137,0.854672,0.382124,0.337352,0.689708,0.309853,0.631380,0.041956,7.957875
9,0.984933,0.699506,0.998885,0.180148,0.580143,0.231087,0.490827,0.313683,5.592193


In [17]:
best = outputs.max()
field = outputs.max() - outputs.min()
print(best, field)

9.78955 4.197356610459804


**5.2 - Next Query**

The global Sobol query returned $y=9.694292$ and did not improve the incumbent value of $9.789550$. However, the first two grid-based queries had improved the initial incumbent from $9.598482$ to $9.789550$, indicating that a promising region had been identified despite the non-improving third grid query and global Sobol query.

The search therefore moved from global exploration to local refinement. The surrogate is upgraded to a Gaussian Process with a scaled Matérn-$5/2$ kernel and a small white-noise component. Expected Improvement and UCB are both evaluated within a local candidate region around the incumbent. For this transition iteration, the UCB proposal is submitted; Expected Improvement becomes the primary query-selection rule from the following iteration, while UCB is retained as a diagnostic comparison.

In [18]:
from scipy.stats import qmc

# Fit GP
d = inputs.shape[1]

kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=np.full(d, 0.20), length_scale_bounds=(1e-2, 3.0), nu=2.5) \
    + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-8, 1e-2))
gp = GaussianProcessRegressor( kernel=kernel, normalize_y=True, n_restarts_optimizer=12, random_state=42)
gp.fit(inputs, outputs)

# Build candidates
x_best = inputs[np.argmax(outputs)]
f_best = np.max(outputs)
radius = np.array([0.12, 0.12, 0.12, 0.12, 0.12, 0.12, 0.12, 0.12])

lb = np.clip(x_best - radius, 0.0, 1.0)
ub = np.clip(x_best + radius, 0.0, 1.0)

m_local = 14

sampler = qmc.Sobol(d=d, scramble=True, seed=123)
X_cand = sampler.random_base2(m_local)
X_cand = lb + (ub - lb) * X_cand

# Remove points too close to existing data to avoid wasted queries
def min_dist_to_data(X, data):
    # squared Euclidean distances
    d2 = ((X[:, None, :] - data[None, :, :]) ** 2).sum(axis=2)
    return np.sqrt(d2.min(axis=1))

min_dist = min_dist_to_data(X_cand, inputs)
X_cand = X_cand[min_dist > 0.05]

# Predict on candidates
y_pred, sigma = gp.predict(X_cand, return_std=True)
sigma = np.maximum(sigma, 1e-12)

# EI Acquisition Query
xi = 0.005
improvement = y_pred - f_best - xi
Z = improvement / sigma
ei = improvement * s.norm.cdf(Z) + sigma * s.norm.pdf(Z)
ei[sigma <= 1e-12] = 0.0

idx_ei = np.argmax(ei)
next_point_ei = X_cand[idx_ei]

print("Current best point:", x_best)
print("Current best value:", f_best)
print("Next point (EI):", next_point_ei)

# UCB Acquisition Query

kappa = 2.5 / np.sqrt(5)   # adaptive exploration
ucb = y_pred + kappa * sigma
idx_ucb = np.argmax(ucb)
next_point_ucb = X_cand[idx_ucb]

print("Next point (UCB):", next_point_ucb)

Current best point: [0.   0.25 0.   0.5  0.75 0.5  0.25 0.5 ]
Current best value: 9.78955
Next point (EI): [0.082969   0.14280299 0.10118516 0.40123035 0.86248022 0.41947722
 0.22507614 0.49829571]
Next point (UCB): [0.11442603 0.13477862 0.11334997 0.38308194 0.85128211 0.49244676
 0.16801044 0.6190627 ]


<h1 style="color:#0000CD;font-size:19px;"">Week 6</h1>

**6.1 - Previous Week's Query Result**

In [19]:
# New Query Point
x_new = np.array([[0.114426, 0.134779, 0.113350, 0.383082, 0.851282, 0.492447, 0.168010, 0.619063]])
y_new = 9.9407381864151

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4','x5','x6','x7','x8']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

Point added!


,x1,x2,x3,x4,x5,x6,x7,x8,y
0,0.604994,0.292215,0.908453,0.355506,0.201669,0.575338,0.310311,0.734281,7.398721
1,0.178007,0.566223,0.994862,0.210325,0.320153,0.707909,0.635384,0.107132,7.005227
2,0.009077,0.811626,0.520520,0.075687,0.265112,0.091652,0.592415,0.367320,8.459482
3,0.506028,0.653730,0.363411,0.177981,0.093728,0.197425,0.755827,0.292472,8.284008
4,0.359909,0.249076,0.495997,0.709215,0.114987,0.289207,0.557295,0.593882,8.606117
5,0.778818,0.003419,0.337983,0.519528,0.820907,0.537247,0.551347,0.660032,8.541748
6,0.908649,0.062250,0.238260,0.766604,0.132336,0.990244,0.688068,0.742496,7.327435
7,0.586371,0.880736,0.745021,0.546035,0.009649,0.748992,0.230907,0.097916,7.299872
8,0.761137,0.854672,0.382124,0.337352,0.689708,0.309853,0.631380,0.041956,7.957875
9,0.984933,0.699506,0.998885,0.180148,0.580143,0.231087,0.490827,0.313683,5.592193


**6.2 - Next Query**

In [20]:
from scipy.stats import qmc

# Fit GP
d = inputs.shape[1]

kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=np.full(d, 0.20), length_scale_bounds=(1e-2, 3.0), nu=2.5) \
    + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-8, 1e-2))
gp = GaussianProcessRegressor( kernel=kernel, normalize_y=True, n_restarts_optimizer=12, random_state=42)
gp.fit(inputs, outputs)

# Build candidates
x_best = inputs[np.argmax(outputs)]
f_best = np.max(outputs)
radius = np.array([0.12, 0.12, 0.12, 0.12, 0.12, 0.12, 0.12, 0.12])

lb = np.clip(x_best - radius, 0.0, 1.0)
ub = np.clip(x_best + radius, 0.0, 1.0)
n_local = 12000
m_local = int(np.ceil(np.log2(n_local)))

sampler = qmc.Sobol(d=d, scramble=True, seed=123)
X_cand = sampler.random_base2(m_local)
X_cand = lb + (ub - lb) * X_cand

# Remove points too close to existing data to avoid wasted queries
def min_dist_to_data(X, data):
    # squared Euclidean distances
    d2 = ((X[:, None, :] - data[None, :, :]) ** 2).sum(axis=2)
    return np.sqrt(d2.min(axis=1))

min_dist = min_dist_to_data(X_cand, inputs)
X_cand = X_cand[min_dist > 0.05]

# Predict on candidates
y_pred, sigma = gp.predict(X_cand, return_std=True)
sigma = np.maximum(sigma, 1e-12)

# EI Acquisition Query
xi = 0.005
improvement = y_pred - f_best - xi
Z = improvement / sigma
ei = improvement * s.norm.cdf(Z) + sigma * s.norm.pdf(Z)
ei[sigma <= 1e-12] = 0.0

idx_ei = np.argmax(ei)
next_point_ei = X_cand[idx_ei]

print("Current best point:", x_best)
print("Current best value:", f_best)
print("Next point (EI):", next_point_ei)

# UCB Acquisition Query

kappa = 2.5 / np.sqrt(6)   # adaptative exploration
ucb = y_pred + kappa * sigma
idx_ucb = np.argmax(ucb)
next_point_ucb = X_cand[idx_ucb]

print("Next point (UCB):", next_point_ucb)

Current best point: [0.114426 0.134779 0.11335  0.383082 0.851282 0.492447 0.16801  0.619063]
Current best value: 9.9407381864151
Next point (EI): [0.15479537 0.01583532 0.13211276 0.28576667 0.96921296 0.47693765
 0.23887379 0.55870773]
Next point (UCB): [0.02169686 0.02640704 0.1266344  0.26872421 0.96537983 0.38110785
 0.15609471 0.5900775 ]


Following the improvement to $9.940738$, the local optimisation policy is re-fitted after every returned observation. Candidate points are generated using the same scrambled Sobol design within a radius-$0.12$ box centred on the current incumbent. Candidates lying within a Euclidean distance of $0.05$ from an existing observation are removed to reduce the risk of submitting redundant queries.

Expected Improvement with $\xi=0.005$ is used as the query-selection rule, while UCB with $\kappa=2.5/\sqrt{t}$ is retained as a diagnostic comparison. The decreasing UCB coefficient progressively reduces the diagnostic emphasis on uncertainty as the query budget is consumed.

<h1 style="color:#0000CD;font-size:19px;"">Week 7</h1>

**7.1 - Previous Week's Query Result**

In [21]:
# New Query Point
x_new = np.array([[0.154795, 0.015835, 0.132113, 0.285767, 0.969213, 0.476938, 0.238874, 0.558708]])
y_new = 9.9395074419221

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4','x5','x6','x7','x8']].to_numpy()
outputs = data['y'].to_numpy()

display(data)

Point added!


,x1,x2,x3,x4,x5,x6,x7,x8,y
0,0.604994,0.292215,0.908453,0.355506,0.201669,0.575338,0.310311,0.734281,7.398721
1,0.178007,0.566223,0.994862,0.210325,0.320153,0.707909,0.635384,0.107132,7.005227
2,0.009077,0.811626,0.520520,0.075687,0.265112,0.091652,0.592415,0.367320,8.459482
3,0.506028,0.653730,0.363411,0.177981,0.093728,0.197425,0.755827,0.292472,8.284008
4,0.359909,0.249076,0.495997,0.709215,0.114987,0.289207,0.557295,0.593882,8.606117
5,0.778818,0.003419,0.337983,0.519528,0.820907,0.537247,0.551347,0.660032,8.541748
6,0.908649,0.062250,0.238260,0.766604,0.132336,0.990244,0.688068,0.742496,7.327435
7,0.586371,0.880736,0.745021,0.546035,0.009649,0.748992,0.230907,0.097916,7.299872
8,0.761137,0.854672,0.382124,0.337352,0.689708,0.309853,0.631380,0.041956,7.957875
9,0.984933,0.699506,0.998885,0.180148,0.580143,0.231087,0.490827,0.313683,5.592193


**7.2 - Next Query**

In [22]:
from scipy.stats import qmc

# Fit GP
d = inputs.shape[1]

kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=np.full(d, 0.20), length_scale_bounds=(1e-2, 3.0), nu=2.5) \
    + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-8, 1e-2))
gp = GaussianProcessRegressor( kernel=kernel, normalize_y=True, n_restarts_optimizer=12, random_state=42)
gp.fit(inputs, outputs)

# Build candidates
x_best = inputs[np.argmax(outputs)]
f_best = np.max(outputs)
radius = np.array([0.12, 0.12, 0.12, 0.12, 0.12, 0.12, 0.12, 0.12])

lb = np.clip(x_best - radius, 0.0, 1.0)
ub = np.clip(x_best + radius, 0.0, 1.0)

m_local = 14

sampler = qmc.Sobol(d=d, scramble=True, seed=123)
X_cand = sampler.random_base2(m_local)
X_cand = lb + (ub - lb) * X_cand

# Remove points too close to existing data to avoid wasted queries
def min_dist_to_data(X, data):
    # squared Euclidean distances
    d2 = ((X[:, None, :] - data[None, :, :]) ** 2).sum(axis=2)
    return np.sqrt(d2.min(axis=1))

min_dist = min_dist_to_data(X_cand, inputs)
X_cand = X_cand[min_dist > 0.05]

# Predict on candidates
y_pred, sigma = gp.predict(X_cand, return_std=True)
sigma = np.maximum(sigma, 1e-12)

# EI Acquisition Query
xi = 0.005
improvement = y_pred - f_best - xi
Z = improvement / sigma
ei = improvement * s.norm.cdf(Z) + sigma * s.norm.pdf(Z)
ei[sigma <= 1e-12] = 0.0

idx_ei = np.argmax(ei)
next_point_ei = X_cand[idx_ei]

print("Current best point:", x_best)
print("Current best value:", f_best)
print("Next point (EI):", next_point_ei)

# UCB Acquisition Query

kappa = 2.5 / np.sqrt(7)   # decreasing exploration weight
ucb = y_pred + kappa * sigma
idx_ucb = np.argmax(ucb)
next_point_ucb = X_cand[idx_ucb]

print("Next point (UCB):", next_point_ucb)

Current best point: [0.114426 0.134779 0.11335  0.383082 0.851282 0.492447 0.16801  0.619063]
Current best value: 9.9407381864151
Next point (EI): [0.00122523 0.07513469 0.18131111 0.26440659 0.93442614 0.48542996
 0.0565489  0.68639896]
Next point (UCB): [0.00122523 0.07513469 0.18131111 0.26440659 0.93442614 0.48542996
 0.0565489  0.68639896]


<h1 style="color:#0000CD;font-size:19px;"">Week 8</h1>

**8.1 - Previous Week's Query Result**

In [23]:
# New Query Point
x_new = np.array([[0.001225, 0.075135, 0.181311, 0.264407, 0.934426, 0.485430, 0.056549, 0.686399]])
y_new = 9.9027444955529

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4','x5','x6','x7','x8']].to_numpy()
outputs = data['y'].to_numpy()

display(data)

Point added!


,x1,x2,x3,x4,x5,x6,x7,x8,y
0,0.604994,0.292215,0.908453,0.355506,0.201669,0.575338,0.310311,0.734281,7.398721
1,0.178007,0.566223,0.994862,0.210325,0.320153,0.707909,0.635384,0.107132,7.005227
2,0.009077,0.811626,0.520520,0.075687,0.265112,0.091652,0.592415,0.367320,8.459482
3,0.506028,0.653730,0.363411,0.177981,0.093728,0.197425,0.755827,0.292472,8.284008
4,0.359909,0.249076,0.495997,0.709215,0.114987,0.289207,0.557295,0.593882,8.606117
5,0.778818,0.003419,0.337983,0.519528,0.820907,0.537247,0.551347,0.660032,8.541748
6,0.908649,0.062250,0.238260,0.766604,0.132336,0.990244,0.688068,0.742496,7.327435
7,0.586371,0.880736,0.745021,0.546035,0.009649,0.748992,0.230907,0.097916,7.299872
8,0.761137,0.854672,0.382124,0.337352,0.689708,0.309853,0.631380,0.041956,7.957875
9,0.984933,0.699506,0.998885,0.180148,0.580143,0.231087,0.490827,0.313683,5.592193


**8.2 - Next Query**

In [24]:
from scipy.stats import qmc

# Fit GP
d = inputs.shape[1]

kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=np.full(d, 0.20), length_scale_bounds=(1e-2, 3.0), nu=2.5) \
    + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-8, 1e-2))
gp = GaussianProcessRegressor( kernel=kernel, normalize_y=True, n_restarts_optimizer=12, random_state=42)
gp.fit(inputs, outputs)

# Build candidates
x_best = inputs[np.argmax(outputs)]
f_best = np.max(outputs)
radius = np.array([0.12, 0.12, 0.12, 0.12, 0.12, 0.12, 0.12, 0.12])

lb = np.clip(x_best - radius, 0.0, 1.0)
ub = np.clip(x_best + radius, 0.0, 1.0)

m_local = 14

sampler = qmc.Sobol(d=d, scramble=True, seed=123)
X_cand = sampler.random_base2(m_local)
X_cand = lb + (ub - lb) * X_cand

# Remove points too close to existing data to avoid wasted queries
def min_dist_to_data(X, data):
    # squared Euclidean distances
    d2 = ((X[:, None, :] - data[None, :, :]) ** 2).sum(axis=2)
    return np.sqrt(d2.min(axis=1))

min_dist = min_dist_to_data(X_cand, inputs)
X_cand = X_cand[min_dist > 0.05]

# Predict on candidates
y_pred, sigma = gp.predict(X_cand, return_std=True)
sigma = np.maximum(sigma, 1e-12)

# EI Acquisition Query
xi = 0.005
improvement = y_pred - f_best - xi
Z = improvement / sigma
ei = improvement * s.norm.cdf(Z) + sigma * s.norm.pdf(Z)
ei[sigma <= 1e-12] = 0.0

idx_ei = np.argmax(ei)
next_point_ei = X_cand[idx_ei]

print("Current best point:", x_best)
print("Current best value:", f_best)
print("Next point (EI):", next_point_ei)

# UCB Acquisition Query

kappa = 2.5 / np.sqrt(8)   # decreasing exploration weight
ucb = y_pred + kappa * sigma
idx_ucb = np.argmax(ucb)
next_point_ucb = X_cand[idx_ucb]

print("Next point (UCB):", next_point_ucb)

Current best point: [0.114426 0.134779 0.11335  0.383082 0.851282 0.492447 0.16801  0.619063]
Current best value: 9.9407381864151
Next point (EI): [0.13603221 0.2378946  0.18029095 0.27569078 0.91957749 0.43701404
 0.20248348 0.70170746]
Next point (UCB): [0.13603221 0.2378946  0.18029095 0.27569078 0.91957749 0.43701404
 0.20248348 0.70170746]


<h1 style="color:#0000CD;font-size:19px;"">Week 9</h1>

**9.1 - Previous Week's Query Result**

In [25]:
# New Query Point
x_new = np.array([[0.136032, 0.237895, 0.180291, 0.275691, 0.919577, 0.437014, 0.202483, 0.701707]])
y_new = 9.9541287497796

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4','x5','x6','x7','x8']].to_numpy()
outputs = data['y'].to_numpy()

display(data)

Point added!


,x1,x2,x3,x4,x5,x6,x7,x8,y
0,0.604994,0.292215,0.908453,0.355506,0.201669,0.575338,0.310311,0.734281,7.398721
1,0.178007,0.566223,0.994862,0.210325,0.320153,0.707909,0.635384,0.107132,7.005227
2,0.009077,0.811626,0.520520,0.075687,0.265112,0.091652,0.592415,0.367320,8.459482
3,0.506028,0.653730,0.363411,0.177981,0.093728,0.197425,0.755827,0.292472,8.284008
4,0.359909,0.249076,0.495997,0.709215,0.114987,0.289207,0.557295,0.593882,8.606117
5,0.778818,0.003419,0.337983,0.519528,0.820907,0.537247,0.551347,0.660032,8.541748
6,0.908649,0.062250,0.238260,0.766604,0.132336,0.990244,0.688068,0.742496,7.327435
7,0.586371,0.880736,0.745021,0.546035,0.009649,0.748992,0.230907,0.097916,7.299872
8,0.761137,0.854672,0.382124,0.337352,0.689708,0.309853,0.631380,0.041956,7.957875
9,0.984933,0.699506,0.998885,0.180148,0.580143,0.231087,0.490827,0.313683,5.592193


**9.2 - Next Query**

In [26]:
from scipy.stats import qmc

# Fit GP
d = inputs.shape[1]

kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=np.full(d, 0.20), length_scale_bounds=(1e-2, 3.0), nu=2.5) \
    + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-8, 1e-2))
gp = GaussianProcessRegressor( kernel=kernel, normalize_y=True, n_restarts_optimizer=12, random_state=42)
gp.fit(inputs, outputs)

# Build candidates
x_best = inputs[np.argmax(outputs)]
f_best = np.max(outputs)
radius = np.array([0.12, 0.12, 0.12, 0.12, 0.12, 0.12, 0.12, 0.12])

lb = np.clip(x_best - radius, 0.0, 1.0)
ub = np.clip(x_best + radius, 0.0, 1.0)
m_local = 14

sampler = qmc.Sobol(d=d, scramble=True, seed=123)
X_cand = sampler.random_base2(m_local)
X_cand = lb + (ub - lb) * X_cand

# Remove points too close to existing data to avoid wasted queries
min_dist = min_dist_to_data(X_cand, inputs)
X_cand = X_cand[min_dist > 0.05]

# Predict on candidates
y_pred, sigma = gp.predict(X_cand, return_std=True)
sigma = np.maximum(sigma, 1e-12)

# EI Acquisition Query
xi = 0.005
improvement = y_pred - f_best - xi
Z = improvement / sigma
ei = improvement * s.norm.cdf(Z) + sigma * s.norm.pdf(Z)
ei[sigma <= 1e-12] = 0.0

idx_ei = np.argmax(ei)
next_point_ei = X_cand[idx_ei]

print("Current best point:", x_best)
print("Current best value:", f_best)
print("Next point (EI):", next_point_ei)

# UCB Acquisition Query

kappa = 2.5 / np.sqrt(9)   # decreasing exploration weight
ucb = y_pred + kappa * sigma
idx_ucb = np.argmax(ucb)
next_point_ucb = X_cand[idx_ucb]

print("Next point (UCB):", next_point_ucb)

Current best point: [0.136032 0.237895 0.180291 0.275691 0.919577 0.437014 0.202483 0.701707]
Current best value: 9.9541287497796
Next point (EI): [0.07916964 0.15598518 0.16275429 0.17354808 0.81915269 0.48316863
 0.22975689 0.5985467 ]
Next point (UCB): [0.07916964 0.15598518 0.16275429 0.17354808 0.81915269 0.48316863
 0.22975689 0.5985467 ]


<h1 style="color:#0000CD;font-size:19px;"">Week 10</h1>

**10.1 - Previous Week's Query Result**

In [27]:
# New Query Point
x_new = np.array([[0.079170, 0.155985, 0.162754, 0.173548, 0.819153, 0.483169, 0.229757, 0.598547]])
y_new = 9.9930855496386

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4','x5','x6','x7','x8']].to_numpy()
outputs = data['y'].to_numpy()

display(data)

Point added!


,x1,x2,x3,x4,x5,x6,x7,x8,y
0,0.604994,0.292215,0.908453,0.355506,0.201669,0.575338,0.310311,0.734281,7.398721
1,0.178007,0.566223,0.994862,0.210325,0.320153,0.707909,0.635384,0.107132,7.005227
2,0.009077,0.811626,0.520520,0.075687,0.265112,0.091652,0.592415,0.367320,8.459482
3,0.506028,0.653730,0.363411,0.177981,0.093728,0.197425,0.755827,0.292472,8.284008
4,0.359909,0.249076,0.495997,0.709215,0.114987,0.289207,0.557295,0.593882,8.606117
5,0.778818,0.003419,0.337983,0.519528,0.820907,0.537247,0.551347,0.660032,8.541748
6,0.908649,0.062250,0.238260,0.766604,0.132336,0.990244,0.688068,0.742496,7.327435
7,0.586371,0.880736,0.745021,0.546035,0.009649,0.748992,0.230907,0.097916,7.299872
8,0.761137,0.854672,0.382124,0.337352,0.689708,0.309853,0.631380,0.041956,7.957875
9,0.984933,0.699506,0.998885,0.180148,0.580143,0.231087,0.490827,0.313683,5.592193


**10.2 - Next Query**

In [28]:
from scipy.stats import qmc

# Fit GP
d = inputs.shape[1]

kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=np.full(d, 0.20), length_scale_bounds=(1e-2, 3.0), nu=2.5) \
    + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-8, 1e-2))
gp = GaussianProcessRegressor( kernel=kernel, normalize_y=True, n_restarts_optimizer=12, random_state=42)
gp.fit(inputs, outputs)

# Build candidates
x_best = inputs[np.argmax(outputs)]
f_best = np.max(outputs)
radius = np.array([0.12, 0.12, 0.12, 0.12, 0.12, 0.12, 0.12, 0.12])

lb = np.clip(x_best - radius, 0.0, 1.0)
ub = np.clip(x_best + radius, 0.0, 1.0)
m_local = 14

sampler = qmc.Sobol(d=d, scramble=True, seed=123)
X_cand = sampler.random_base2(m_local)
X_cand = lb + (ub - lb) * X_cand

# Remove points too close to existing data to avoid wasted queries
min_dist = min_dist_to_data(X_cand, inputs)
X_cand = X_cand[min_dist > 0.05]

# Predict on candidates
y_pred, sigma = gp.predict(X_cand, return_std=True)
sigma = np.maximum(sigma, 1e-12)

# EI Acquisition Query
xi = 0.005
improvement = y_pred - f_best - xi
Z = improvement / sigma
ei = improvement * s.norm.cdf(Z) + sigma * s.norm.pdf(Z)
ei[sigma <= 1e-12] = 0.0

idx_ei = np.argmax(ei)
next_point_ei = X_cand[idx_ei]

print("Current best point:", x_best)
print("Current best value:", f_best)
print("Next point (EI):", next_point_ei)

# UCB Acquisition Query

kappa = 2.5 / np.sqrt(10)   # decreasing exploration weight
ucb = y_pred + kappa * sigma
idx_ucb = np.argmax(ucb)
next_point_ucb = X_cand[idx_ucb]

print("Next point (UCB):", next_point_ucb)

Current best point: [0.07917  0.155985 0.162754 0.173548 0.819153 0.483169 0.229757 0.598547]
Current best value: 9.9930855496386
Next point (EI): [0.14424701 0.05065961 0.21737677 0.05537091 0.73446882 0.38726149
 0.19268296 0.63735576]
Next point (UCB): [0.15283065 0.0659738  0.19564525 0.08751835 0.75925512 0.47485076
 0.17927404 0.60962783]


<h1 style="color:#0000CD;font-size:19px;"">Week 11</h1>

**11.1 - Previous Week's Query Result**

In [29]:
# New Query Point
x_new = np.array([[0.144247, 0.050660, 0.217377, 0.055371, 0.734469, 0.387261, 0.192683, 0.637356]])
y_new = 9.9392532402009

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4','x5','x6','x7','x8']].to_numpy()
outputs = data['y'].to_numpy()

display(data)

Point added!


,x1,x2,x3,x4,x5,x6,x7,x8,y
0,0.604994,0.292215,0.908453,0.355506,0.201669,0.575338,0.310311,0.734281,7.398721
1,0.178007,0.566223,0.994862,0.210325,0.320153,0.707909,0.635384,0.107132,7.005227
2,0.009077,0.811626,0.520520,0.075687,0.265112,0.091652,0.592415,0.367320,8.459482
3,0.506028,0.653730,0.363411,0.177981,0.093728,0.197425,0.755827,0.292472,8.284008
4,0.359909,0.249076,0.495997,0.709215,0.114987,0.289207,0.557295,0.593882,8.606117
5,0.778818,0.003419,0.337983,0.519528,0.820907,0.537247,0.551347,0.660032,8.541748
6,0.908649,0.062250,0.238260,0.766604,0.132336,0.990244,0.688068,0.742496,7.327435
7,0.586371,0.880736,0.745021,0.546035,0.009649,0.748992,0.230907,0.097916,7.299872
8,0.761137,0.854672,0.382124,0.337352,0.689708,0.309853,0.631380,0.041956,7.957875
9,0.984933,0.699506,0.998885,0.180148,0.580143,0.231087,0.490827,0.313683,5.592193


**11.2 - Next Query**

In [30]:
from scipy.stats import qmc

# Fit GP
d = inputs.shape[1]

kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=np.full(d, 0.20), length_scale_bounds=(1e-2, 3.0), nu=2.5) \
    + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-8, 1e-2))
gp = GaussianProcessRegressor( kernel=kernel, normalize_y=True, n_restarts_optimizer=12, random_state=42)
gp.fit(inputs, outputs)

# Build candidates
x_best = inputs[np.argmax(outputs)]
f_best = np.max(outputs)
radius = np.array([0.12, 0.12, 0.12, 0.12, 0.12, 0.12, 0.12, 0.12])

lb = np.clip(x_best - radius, 0.0, 1.0)
ub = np.clip(x_best + radius, 0.0, 1.0)
n_local = 12000
m_local = int(np.ceil(np.log2(n_local)))

sampler = qmc.Sobol(d=d, scramble=True, seed=123)
X_cand = sampler.random_base2(m_local)
X_cand = lb + (ub - lb) * X_cand

# Remove points too close to existing data to avoid wasted queries
min_dist = min_dist_to_data(X_cand, inputs)
X_cand = X_cand[min_dist > 0.05]

# Predict on candidates
y_pred, sigma = gp.predict(X_cand, return_std=True)
sigma = np.maximum(sigma, 1e-12)

# EI Acquisition Query
xi = 0.005
improvement = y_pred - f_best - xi
Z = improvement / sigma
ei = improvement * s.norm.cdf(Z) + sigma * s.norm.pdf(Z)
ei[sigma <= 1e-12] = 0.0

idx_ei = np.argmax(ei)
next_point_ei = X_cand[idx_ei]

print("Current best point:", x_best)
print("Current best value:", f_best)
print("Next point (EI):", next_point_ei)

# UCB Acquisition Query

kappa = 2.5 / np.sqrt(11)   # decreasing exploration weight
ucb = y_pred + kappa * sigma
idx_ucb = np.argmax(ucb)
next_point_ucb = X_cand[idx_ucb]

print("Next point (UCB):", next_point_ucb)

Current best point: [0.07917  0.155985 0.162754 0.173548 0.819153 0.483169 0.229757 0.598547]
Current best value: 9.9930855496386
Next point (EI): [0.10082087 0.16351123 0.15764727 0.08993732 0.9213392  0.59710547
 0.20913272 0.7108336 ]
Next point (UCB): [0.05841262 0.15747237 0.14059258 0.06684729 0.91400039 0.56128831
 0.20860545 0.6813713 ]


<h1 style="color:#0000CD;font-size:19px;"">Week 12</h1>

**12.1 - Previous Week's Query Result**

In [31]:
# New Query Point
x_new = np.array([[0.100821, 0.163511, 0.157647, 0.089937, 0.921339, 0.597105, 0.209133, 0.710834]])
y_new = 9.9757292725819

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4','x5','x6','x7','x8']].to_numpy()
outputs = data['y'].to_numpy()

display(data)

Point added!


,x1,x2,x3,x4,x5,x6,x7,x8,y
0,0.604994,0.292215,0.908453,0.355506,0.201669,0.575338,0.310311,0.734281,7.398721
1,0.178007,0.566223,0.994862,0.210325,0.320153,0.707909,0.635384,0.107132,7.005227
2,0.009077,0.811626,0.520520,0.075687,0.265112,0.091652,0.592415,0.367320,8.459482
3,0.506028,0.653730,0.363411,0.177981,0.093728,0.197425,0.755827,0.292472,8.284008
4,0.359909,0.249076,0.495997,0.709215,0.114987,0.289207,0.557295,0.593882,8.606117
5,0.778818,0.003419,0.337983,0.519528,0.820907,0.537247,0.551347,0.660032,8.541748
6,0.908649,0.062250,0.238260,0.766604,0.132336,0.990244,0.688068,0.742496,7.327435
7,0.586371,0.880736,0.745021,0.546035,0.009649,0.748992,0.230907,0.097916,7.299872
8,0.761137,0.854672,0.382124,0.337352,0.689708,0.309853,0.631380,0.041956,7.957875
9,0.984933,0.699506,0.998885,0.180148,0.580143,0.231087,0.490827,0.313683,5.592193


**12.2 - Next Query**

In [32]:
from scipy.stats import qmc

# Fit GP
d = inputs.shape[1]

kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=np.full(d, 0.20), length_scale_bounds=(1e-2, 3.0), nu=2.5) \
    + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-8, 1e-2))
gp = GaussianProcessRegressor( kernel=kernel, normalize_y=True, n_restarts_optimizer=12, random_state=42)
gp.fit(inputs, outputs)

# Build candidates
x_best = inputs[np.argmax(outputs)]
f_best = np.max(outputs)
radius = np.array([0.12, 0.12, 0.12, 0.12, 0.12, 0.12, 0.12, 0.12])

lb = np.clip(x_best - radius, 0.0, 1.0)
ub = np.clip(x_best + radius, 0.0, 1.0)
n_local = 12000
m_local = int(np.ceil(np.log2(n_local)))

sampler = qmc.Sobol(d=d, scramble=True, seed=123)
X_cand = sampler.random_base2(m_local)
X_cand = lb + (ub - lb) * X_cand

# Remove points too close to existing data to avoid wasted queries
min_dist = min_dist_to_data(X_cand, inputs)
X_cand = X_cand[min_dist > 0.05]

# Predict on candidates
y_pred, sigma = gp.predict(X_cand, return_std=True)
sigma = np.maximum(sigma, 1e-12)

# EI Acquisition Query
xi = 0.005
improvement = y_pred - f_best - xi
Z = improvement / sigma
ei = improvement * s.norm.cdf(Z) + sigma * s.norm.pdf(Z)
ei[sigma <= 1e-12] = 0.0

idx_ei = np.argmax(ei)
next_point_ei = X_cand[idx_ei]

print("Current best point:", x_best)
print("Current best value:", f_best)
print("Next point (EI):", next_point_ei)

# UCB Acquisition Query

kappa = 2.5 / np.sqrt(12)   # decreasing exploration weight
ucb = y_pred + kappa * sigma
idx_ucb = np.argmax(ucb)
next_point_ucb = X_cand[idx_ucb]

print("Next point (UCB):", next_point_ucb)

Current best point: [0.07917  0.155985 0.162754 0.173548 0.819153 0.483169 0.229757 0.598547]
Current best value: 9.9930855496386
Next point (EI): [0.05274331 0.14501284 0.0815718  0.05547984 0.90555815 0.37240466
 0.24165334 0.6876209 ]
Next point (UCB): [0.10213229 0.2160055  0.17785426 0.11688321 0.92327578 0.48765563
 0.22928312 0.49842326]


<h1 style="color:#0000CD;font-size:19px;"">Week 13</h1>

**13.1 - Previous Week's Query Result**

In [33]:
# New Query Point
x_new = np.array([[0.052743, 0.145013, 0.081572, 0.055480, 0.905558, 0.372405, 0.241653, 0.687621]])
y_new = 9.9534494192919

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4','x5','x6','x7','x8']].to_numpy()
outputs = data['y'].to_numpy()

display(data)

Point added!


,x1,x2,x3,x4,x5,x6,x7,x8,y
0,0.604994,0.292215,0.908453,0.355506,0.201669,0.575338,0.310311,0.734281,7.398721
1,0.178007,0.566223,0.994862,0.210325,0.320153,0.707909,0.635384,0.107132,7.005227
2,0.009077,0.811626,0.520520,0.075687,0.265112,0.091652,0.592415,0.367320,8.459482
3,0.506028,0.653730,0.363411,0.177981,0.093728,0.197425,0.755827,0.292472,8.284008
4,0.359909,0.249076,0.495997,0.709215,0.114987,0.289207,0.557295,0.593882,8.606117
5,0.778818,0.003419,0.337983,0.519528,0.820907,0.537247,0.551347,0.660032,8.541748
6,0.908649,0.062250,0.238260,0.766604,0.132336,0.990244,0.688068,0.742496,7.327435
7,0.586371,0.880736,0.745021,0.546035,0.009649,0.748992,0.230907,0.097916,7.299872
8,0.761137,0.854672,0.382124,0.337352,0.689708,0.309853,0.631380,0.041956,7.957875
9,0.984933,0.699506,0.998885,0.180148,0.580143,0.231087,0.490827,0.313683,5.592193


**13.2 - Next Query**

With one query remaining, the EI offset is reduced from $\xi=0.005$ to $\xi=0$, prioritising expected improvement around the incumbent over additional exploratory displacement.

In [34]:
from scipy.stats import qmc

# Fit GP
d = inputs.shape[1]

kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=np.full(d, 0.20), length_scale_bounds=(1e-2, 3.0), nu=2.5) \
    + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-8, 1e-2))
gp = GaussianProcessRegressor( kernel=kernel, normalize_y=True, n_restarts_optimizer=12, random_state=42)
gp.fit(inputs, outputs)

# Build candidates
x_best = inputs[np.argmax(outputs)]
f_best = np.max(outputs)
radius = np.array([0.12, 0.12, 0.12, 0.12, 0.12, 0.12, 0.12, 0.12])

lb = np.clip(x_best - radius, 0.0, 1.0)
ub = np.clip(x_best + radius, 0.0, 1.0)
m_local = 14

sampler = qmc.Sobol(d=d, scramble=True, seed=123)
X_cand = sampler.random_base2(m_local)
X_cand = lb + (ub - lb) * X_cand

# Remove points too close to existing data to avoid wasted queries
min_dist = min_dist_to_data(X_cand, inputs)
X_cand = X_cand[min_dist > 0.05]

# Predict on candidates
y_pred, sigma = gp.predict(X_cand, return_std=True)
sigma = np.maximum(sigma, 1e-12)

# EI Acquisition Query
xi = 0.00
improvement = y_pred - f_best - xi
Z = improvement / sigma
ei = improvement * s.norm.cdf(Z) + sigma * s.norm.pdf(Z)
ei[sigma <= 1e-12] = 0.0

idx_ei = np.argmax(ei)
next_point_ei = X_cand[idx_ei]

print("Current best point:", x_best)
print("Current best value:", f_best)
print("Next point (EI):", next_point_ei)

# UCB Acquisition Query

kappa = 2.5 / np.sqrt(13)   # decreasing exploration weight
ucb = y_pred + kappa * sigma
idx_ucb = np.argmax(ucb)
next_point_ucb = X_cand[idx_ucb]

print("Next point (UCB):", next_point_ucb)

Current best point: [0.07917  0.155985 0.162754 0.173548 0.819153 0.483169 0.229757 0.598547]
Current best value: 9.9930855496386
Next point (EI): [0.10213229 0.2160055  0.17785426 0.11688321 0.92327578 0.48765563
 0.22928312 0.49842326]
Next point (UCB): [0.10213229 0.2160055  0.17785426 0.11688321 0.92327578 0.48765563
 0.22928312 0.49842326]


<h1 style="color:#0000CD;font-size:19px;"">14. Final Result</h1>

**14.1 -  Previous Week's Query Result**

In [35]:
# Queried Point result
x_new = np.array([[0.102132, 0.216006, 0.177854, 0.116883, 0.923276, 0.487656, 0.229283, 0.498423]])
y_new = 9.9771697281841

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

Point added!


,x1,x2,x3,x4,x5,x6,x7,x8,y
0,0.604994,0.292215,0.908453,0.355506,0.201669,0.575338,0.310311,0.734281,7.398721
1,0.178007,0.566223,0.994862,0.210325,0.320153,0.707909,0.635384,0.107132,7.005227
2,0.009077,0.811626,0.520520,0.075687,0.265112,0.091652,0.592415,0.367320,8.459482
3,0.506028,0.653730,0.363411,0.177981,0.093728,0.197425,0.755827,0.292472,8.284008
4,0.359909,0.249076,0.495997,0.709215,0.114987,0.289207,0.557295,0.593882,8.606117
5,0.778818,0.003419,0.337983,0.519528,0.820907,0.537247,0.551347,0.660032,8.541748
6,0.908649,0.062250,0.238260,0.766604,0.132336,0.990244,0.688068,0.742496,7.327435
7,0.586371,0.880736,0.745021,0.546035,0.009649,0.748992,0.230907,0.097916,7.299872
8,0.761137,0.854672,0.382124,0.337352,0.689708,0.309853,0.631380,0.041956,7.957875
9,0.984933,0.699506,0.998885,0.180148,0.580143,0.231087,0.490827,0.313683,5.592193


The sequential Bayesian optimisation process improved the best observed objective value from $9.598482$ in the initial sample to $9.993086$ after 13 submitted queries. This represents an absolute improvement of $0.394604$ and a relative increase of approximately $4.1%$.

The search progressed from coarse Cartesian-grid UCB exploration to broader Sobol-based UCB candidate generation, followed by local Expected Improvement refinement once a promising region had been identified. Several later queries did not improve the incumbent but provided additional observations around the high-value region.

The overall incumbent was identified during Week 10 at

$$
[0.079170,\ 0.155985,\ 0.162754,\ 0.173548,\ 0.819153,\ 0.483169,\ 0.229757,\ 0.598547],
$$

with an observed objective value of $9.993086$. The final submitted query returned $9.977170$ and therefore did not exceed the incumbent.

Although the hidden nature of the objective prevents any claim that the incumbent is the global optimum, the observed progression demonstrates that the adaptive strategy successfully identified and refined a consistently high-value local region within the available query budget.